# ManūAI · 01 — Data: index, grouped split, features

Thin, interactive front-end over the `manuai/` package (the package holds the logic so training,
evaluation and the app all share it).

1. Build `data/index.csv` from the archive metadata with the **grouped** train/val/test split
2. Look at class balance and what the √-frequency sampler will feed the model
3. Listen to and look at the 5 s features for each class
4. Build the feature cache (guarded — the full cache is ~15 min and ~34 GB)

In [ ]:
%load_ext autoreload
%autoreload 2
import json, os
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, Audio, display

from manuai import features as F
from manuai.config import REPO_ROOT, data_dir
from manuai.labels import class_names
os.chdir(REPO_ROOT)
CLASSES = class_names()
print(f"repo: {REPO_ROOT}\nclasses ({len(CLASSES)}): {', '.join(CLASSES)}")

## 1 · Index and grouped split
Re-running is cheap (~20 s) and deterministic (seed 21).

In [ ]:
from manuai import build_index
REBUILD_INDEX = not (data_dir(create=True) / "index.csv").exists()
if REBUILD_INDEX:
    build_index.main([])
index = pd.read_csv(data_dir() / "index.csv")
print(f"{len(index):,} clips · {index.recording_id.nunique():,} recordings · {index.group.nunique():,} groups")
display(Markdown((data_dir() / "split_summary.md").read_text()))

## 2 · Class balance and sampling

In [ ]:
counts = (index.groupby(["label", "split"]).size().unstack("split")
          .reindex(CLASSES)[["train", "val", "test"]].fillna(0))
ax = counts.plot.barh(figsize=(9, 7), logx=True, width=0.8)
ax.invert_yaxis(); ax.set_xlabel("clips (log scale)"); ax.set_title("Clips per class and split")
plt.tight_layout(); plt.show()

recs = index[index.split == "test"].groupby("label").recording_id.nunique().reindex(CLASSES)
print("Test recordings per class (smallest first):")
print(recs.sort_values().head(8).to_string())

In [ ]:
# What the training sampler draws: class share ∝ n_c^alpha (alpha=0.5 → square root)
train_counts = index[index.split == "train"].groupby("label").size().reindex(CLASSES)
share = pd.DataFrame({
    "natural": train_counts / train_counts.sum(),
    "sqrt (alpha=0.5)": train_counts**0.5 / (train_counts**0.5).sum(),
    "balanced (alpha=0)": pd.Series(1 / len(CLASSES), index=CLASSES),
})
ax = share.plot.barh(figsize=(9, 7), width=0.8)
ax.invert_yaxis(); ax.set_xlabel("share of training draws"); plt.tight_layout(); plt.show()

## 3 · Listen and look
One test clip per class through the exact feature code used for training (`manuai.features`).

In [ ]:
rng = np.random.default_rng(21)
examples = (index[index.split == "test"].groupby("label").sample(1, random_state=21)
            .set_index("label").reindex(CLASSES))
n = len(CLASSES); cols = 4; rows = -(-n // cols)
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 2.4 * rows))
for ax, (label, row) in zip(axes.flat, examples.iterrows()):
    x = F.features_from_file(REPO_ROOT / row.path)
    ax.imshow(x[0].T, origin="lower", aspect="auto", cmap="magma")
    ax.set_title(f"{label} · {row.source}", fontsize=9); ax.axis("off")
for ax in axes.flat[n:]:
    ax.axis("off")
plt.suptitle("5 s Kaldi fbank (128 mel, 16 kHz) — what the model sees", y=1.0); plt.tight_layout(); plt.show()

In [ ]:
LISTEN_TO = "Tui"   # any class name
row = examples.loc[LISTEN_TO]
print(row.path)
display(Audio(str(REPO_ROOT / row.path)))

## 4 · Feature cache

| cache | what | cost |
|---|---|---|
| `features_smoke` | 200 train / 40 val+test clips per class | ~1 min, 0.9 GB |
| `features` | everything | ~15 min, ~34 GB |

Both are guarded so *Run All* never starts a big job by accident.

In [ ]:
from manuai import preprocess
BUILD_SMOKE = False   # set True to (re)build the small balanced subset
BUILD_FULL = False    # set True to build the full cache (~15 min, ~34 GB) — the overnight job does this
if BUILD_SMOKE:
    preprocess.main(["--name", "features_smoke", "--per-class", "200", "--per-class-eval", "40", "--force"])
if BUILD_FULL:
    preprocess.main(["--name", "features"])

In [ ]:
for name in ("features_smoke", "features"):
    meta_path = data_dir() / name / "meta.json"
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        size = sum(p.stat().st_size for p in (data_dir() / name).glob("*.npy")) / 1e9
        print(f"✅ {name}: {json.dumps(meta['counts'])} · {size:.1f} GB · stats {meta.get('train_stats')}")
    else:
        print(f"— {name}: not built")